In [17]:
import functools
import os
import shelve

import numpy as np
from matplotlib import pyplot as plt

from synthetic.model import do_synthetic, normalise, trim, rmse, Layer, resample

In [18]:
DATA_DIR = "../data"
datafile = functools.partial(os.path.join, DATA_DIR)


def prf(evt_id: str):
    return np.load(datafile(f"prfs/{evt_id}.npy"))

In [19]:
mean_rf_t, mean_rf_y = np.load(datafile("mean_rf.npy"))
mx, my = trim(0, 10, mean_rf_t, mean_rf_y)

models = shelve.open(datafile("models"))

In [20]:
fig, axs = plt.subplots(5, 2, figsize=(12, 16))

for x, i in enumerate(("mean_", "best_")):
    axs[0][x].plot(mx, my, label="mean")
    axs[0][x].set_title("reference")
    for y, k in enumerate(filter(lambda z: z.startswith(i), models.keys())):
        ll = models[k]
        xx, yy, model = do_synthetic(ll)
        xx, yy = trim(0, 10, *normalise(xx, yy))
        # err = rmse(xx, yy, mx, my)
        axs[y + 1][x].plot(xx, yy, label=k)
        axs[y + 1][x].set_title(k)

# axs.legend()

In [21]:
import pickle

print(pickle.DEFAULT_PROTOCOL)

## Comparison of S1222a & 4 layers 

In [22]:
shi_et_al_4a = [
    Layer(thickness=2000, v_p=5000, v_s=3000, rho=2500),
    Layer(thickness=8000, v_p=3800, v_s=1850, rho=2304),
    Layer(thickness=12000, v_p=4500, v_s=2800, rho=2570),
    Layer(thickness=24000, v_p=6220, v_s=3750, rho=2863),
    Layer(thickness=0, v_p=7670, v_s=4330, rho=3446),
]
xx, yy, model = do_synthetic(shi_et_al_4a)
model.plot_profile()

In [23]:
mx, my = resample(mx, my, 0, 10, 20)
xx, yy = resample(xx, yy, 0, 10, 20)

In [24]:
f2, a2 = plt.subplots(1, 1)
a2.plot(xx, yy, label="synth")
a2.plot(mx, my, label="prf")
a2.legend()